<a href="https://colab.research.google.com/github/Smyles019/html-login-form-detector/blob/feat%2Fxgboost/notebooks/02_xgboost_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Smyles019/html-login-form-detector.git

Cloning into 'html-login-form-detector'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 110 (delta 37), reused 38 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 390.13 KiB | 2.12 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [2]:
import os
import shutil
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from bs4 import BeautifulSoup
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [3]:
# 1. Reset and recreate processed directory
processed_path = 'html-login-form-detector/data/processed'
if os.path.exists(processed_path):
    if os.path.isfile(processed_path):
        os.remove(processed_path)
    elif os.path.isdir(processed_path):
        shutil.rmtree(processed_path)
os.makedirs(processed_path, exist_ok=True)

# 2. Comprehensive label schema map
raw_path = 'html-login-form-detector/data/raw/Dataset 1/'
label_mapping = {
    'LOGIN_FORM_MALICIOUS': 'LOGIN_FORM',
    'TEST_LOGIN_FORM': 'LOGIN_FORM',
    'LOGIN_FORM': 'LOGIN_FORM',
    'TEST_NO_FORM': 'NO_FORM',
    'NO_FORM': 'NO_FORM'
}

def load_and_clean(file_form, file_no_form):
    df1 = pd.read_csv(raw_path + file_form)
    df2 = pd.read_csv(raw_path + file_no_form)

    # Combine & deduplicate signatures
    df = pd.concat([df1, df2], ignore_index=True).drop_duplicates(subset=['html_signature'])

    # Fill text gaps and map target labels
    df['html_signature'] = df['html_signature'].fillna('')
    df['clean_label'] = df['label'].map(label_mapping).fillna(df['label'])

    return df.dropna(subset=['clean_label']).reset_index(drop=True)

# 3. Clean and save output datasets
df_train = load_and_clean('train_login_form.csv', 'train_no_form.csv')
df_val = load_and_clean('validation_login_form.csv', 'validation_no_form.csv')

df_train.to_csv(f'{processed_path}/train_cleaned.csv', index=False)
df_val.to_csv(f'{processed_path}/validation_cleaned.csv', index=False)

print("Unique labels in Train:", df_train['clean_label'].unique())
print("Unique labels in Val:  ", df_val['clean_label'].unique())
print(f"Data cleaned & saved! Train: {len(df_train)} rows | Val: {len(df_val)} rows")

Unique labels in Train: ['LOGIN_FORM' 'NO_FORM']
Unique labels in Val:   ['LOGIN_FORM' 'NO_FORM']
Data cleaned & saved! Train: 1141 rows | Val: 205 rows


In [16]:
# Upgraded Cell 4: Abstract Tag-Topology Extraction
import re
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import RobustScaler, LabelEncoder
import pandas as pd
import numpy as np

def extract_structural_topology(html_series):
    metrics = []
    for s in html_series:
        s = str(s).lower()

        # 1. Direct Tag Counts
        c_form = s.count('(form')
        c_input = s.count('(input')
        c_button = s.count('(button')
        c_label = s.count('(label')
        c_a = s.count('(a')
        c_div = s.count('(div')

        # 2. Sequential & Structural Patterns
        has_form_input = 1 if re.search(r'\(form.*\(input', s) else 0
        has_multi_input = 1 if c_input >= 2 else 0
        has_input_button = 1 if re.search(r'\(input.*\(button', s) else 0
        has_form_button = 1 if re.search(r'\(form.*\(button', s) else 0

        # 3. Ratio Indicators
        input_to_div_ratio = c_input / (c_div + 1)

        # 4. Maximum Nesting Depth (counting consecutive opening parenthesis)
        max_depth = 0
        curr_depth = 0
        for char in s:
            if char == '(':
                curr_depth += 1
                if curr_depth > max_depth:
                    max_depth = curr_depth
            elif char == ')':
                curr_depth = max(0, curr_depth - 1)

        metrics.append([
            c_form, c_input, c_button, c_label, c_a, c_div,
            has_form_input, has_multi_input, has_input_button, has_form_button,
            input_to_div_ratio, max_depth
        ])
    return np.array(metrics)

# Load Data
df_train = pd.read_csv('html-login-form-detector/data/processed/train_cleaned.csv')
df_val = pd.read_csv('html-login-form-detector/data/processed/validation_cleaned.csv')

df_train['html_signature'] = df_train['html_signature'].fillna('')
df_val['html_signature'] = df_val['html_signature'].fillna('')

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(df_train['clean_label'].astype(str))
y_val = label_encoder.transform(df_val['clean_label'].astype(str))

# 1. Structural Tag Sequences via Word N-Grams
# Using custom token pattern to treat tag tokens like '(form', '(input', ')' as distinct words
seq_vectorizer = CountVectorizer(
    token_pattern=r'\([a-z0-9]+|\)',
    ngram_range=(1, 5),
    max_features=4000
)
X_train_seq = seq_vectorizer.fit_transform(df_train['html_signature'])
X_val_seq = seq_vectorizer.transform(df_val['html_signature'])

# 2. Extract Topological Metrics
X_train_top_raw = extract_structural_topology(df_train['html_signature'])
X_val_top_raw = extract_structural_topology(df_val['html_signature'])

scaler = RobustScaler()
X_train_top = scaler.fit_transform(X_train_top_raw)
X_val_top = scaler.transform(X_val_top_raw)

# Combine Structural Counts & Sequence Sub-Trees
X_train = hstack([X_train_top, X_train_seq]).tocsr()
X_val = hstack([X_val_top, X_val_seq]).tocsr()

print(f"Topology Matrix Ready! X_train shape: {X_train.shape}")

Topology Matrix Ready! X_train shape: (1141, 4012)


In [17]:
# Cell 5: Upgraded Hyperparameters for Aggressive Recall
model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=6,                 # Increased depth to capture multi-input/JS login edge cases
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=3.5,        # Heavy penalty on false negatives (missing LOGIN_FORM)
    early_stopping_rounds=40,
    random_state=42,
    eval_metric='auc'            # Optimizes ranking performance directly
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

[0]	validation_0-auc:0.72144
[50]	validation_0-auc:0.89558
[100]	validation_0-auc:0.91871
[150]	validation_0-auc:0.92947
[200]	validation_0-auc:0.93699
[250]	validation_0-auc:0.93708
[300]	validation_0-auc:0.93775
[315]	validation_0-auc:0.93775


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=40,
              enable_categorical=True, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.02, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

In [18]:
# Probabilities for Class 0 (LOGIN_FORM) and Class 1 (NO_FORM)
y_proba_login = model.predict_proba(X_val)[:, 0]
y_proba_no_form = model.predict_proba(X_val)[:, 1]

# Calibrated decision boundary for security focus
login_threshold = 0.20
y_pred_adjusted = np.where(y_proba_login >= login_threshold, 0, 1)

print(f"=== Adjusted Evaluation (LOGIN_FORM Threshold = {login_threshold}) ===")
print(classification_report(y_val, y_pred_adjusted, target_names=label_encoder.classes_))

# Metric checks
roc_score = roc_auc_score(y_val, y_proba_no_form)
print(f"Corrected ROC-AUC Score: {roc_score:.4f}")

# Confusion Matrix display
cm = confusion_matrix(y_val, y_pred_adjusted)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {c}" for c in label_encoder.classes_],
    columns=[f"Pred {c}" for c in label_encoder.classes_]
)
print("\n=== Confusion Matrix ===")
print(cm_df)

=== Adjusted Evaluation (LOGIN_FORM Threshold = 0.2) ===
              precision    recall  f1-score   support

  LOGIN_FORM       0.93      0.75      0.83       103
     NO_FORM       0.79      0.94      0.86       102

    accuracy                           0.84       205
   macro avg       0.86      0.84      0.84       205
weighted avg       0.86      0.84      0.84       205

Corrected ROC-AUC Score: 0.9383

=== Confusion Matrix ===
                   Pred LOGIN_FORM  Pred NO_FORM
Actual LOGIN_FORM               77            26
Actual NO_FORM                   6            96


In [7]:
# Save model, vectorizer, scaler, and label encoder
os.makedirs('html-login-form-detector/models', exist_ok=True)
joblib.dump(model, 'html-login-form-detector/models/xgboost_model.pkl')
joblib.dump(vectorizer, 'html-login-form-detector/models/vectorizer.pkl')
joblib.dump(scaler, 'html-login-form-detector/models/scaler.pkl')
joblib.dump(label_encoder, 'html-login-form-detector/models/label_encoder.pkl')

print("Complete model pipeline successfully saved to 'html-login-form-detector/models/'!")

Complete model pipeline successfully saved to 'html-login-form-detector/models/'!


In [15]:
# Diagnostic: Inspect the False Negatives (Missed Login Forms)
false_negatives_mask = (y_val == 0) & (y_pred_adjusted == 1)
fn_signatures = df_val.loc[false_negatives_mask, 'html_signature']

print(f"Total Missed Login Forms: {len(fn_signatures)}\n")
print("=== Sample of Missed HTML Signatures ===")
for i, sig in enumerate(fn_signatures.head(5), 1):
    print(f"\n--- Missed Form #{i} ---")
    print(sig[:300]) # Print first 300 characters

Total Missed Login Forms: 27

=== Sample of Missed HTML Signatures ===

--- Missed Form #1 ---
(body(div(div)(img)(div)(div(img))(div(div(div(div)(div)(div)(div)(div)(div)(label)(input)(div)(label)(img)(input)(div)(div)(div)(div)(center)))))(div)(script)(script)(script)(script)(script)(script))

--- Missed Form #2 ---
(body(div(style)(img)(span(img)))(div(div(div(div(div(div)))(div(div(div(div(div(div(section(main(div(div(div(div(div))(div(div(form(div(div(span))(div(div(label(span)(input))(div)))(div(div(label(span)(input))(div)))(div(button(div)))(div(div(div)(div)(div)))(div(button(span)(span))))(a(span))))))(

--- Missed Form #3 ---
(body(main(div(div(img)(div(form(input)(input)(button)))(span)(div(a(img)(span)))(a))(div(span)(div(a(img))(a(img)))))(div(img)))(footer(ul(li(a))(li(a))(li(a))(li(a))(li(a))(li(a))(li(a))(li(a))(li(a))(li(a))(li(a))(li(a)))(span))(script)(script)(script)(script)(script)(script)(script)(script)(next

--- Missed Form #4 ---
(body(div(header(div(div(img))